# Imports and Libraries

## Installs

Dataset and code for downloading found:
https://huggingface.co/docs/datasets/v1.13.2/load_hub.html

In [1]:
# Commented out to stop multiple runs when running entire notebook
#%pip install -U datasets

## Imports

`load_dataset` - For downloading sst2 dataset

`DatasetDict` - For creating project dataset splits

`pd` - For usage of DataFrame type

`AutoTokenizer` - For tokenisation of datasets required for transformer based models

`np` - Used in evaluation metrics for finding highest model output score

`accuracy_score` - Used to calculate overall accuracy of model

`precision_recall_fscore_support` - Used to calculate precision, recall, F1 and support of model

`` - 

`` - 

`` - 


In [18]:
from datasets import load_dataset, DatasetDict
import pandas as pd
from transformers import AutoTokenizer
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Dataset

`dataframe_check` function:

- Takes:
  - `dataframe`: a dataFrame containing training, validation and testing data


This function performs a couple of manual checks on the dataset for inspection by the user. For example, printing the first few values and checking the balance

In [3]:
def dataframe_check(dataframe):
    # Check first couple values
    print("Dataframe head:")
    print(dataframe.head())
    print("\n-----------------------------------")
    print("\nBalance of dataset")
    print(dataframe["label"].value_counts())
    print(dataframe["label"].value_counts(normalize=True))

Load the sst2 dataset.

In [4]:
sst2 = load_dataset("glue", "sst2")

README.md: 0.00B [00:00, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

Convert to a dataframe and call the `dataframe_check` function to print some infomation about the training data

In [5]:
train_df = pd.DataFrame(sst2["train"])
dataframe_check(train_df)

Dataframe head:
                                            sentence  label  idx
0       hide new secretions from the parental units       0    0
1               contains no wit , only labored gags       0    1
2  that loves its characters and communicates som...      1    2
3  remains utterly satisfied to remain the same t...      0    3
4  on the worst revenge-of-the-nerds clichés the ...      0    4

-----------------------------------

Balance of dataset
label
1    37569
0    29780
Name: count, dtype: int64
label
1    0.557826
0    0.442174
Name: proportion, dtype: float64


# Create datset splits

The SST-2 has hidden labels for test data so only training and validation data will be taken into account. Training data will be split into training and testing.

In [6]:
train_test_split = sst2["train"].train_test_split(
    test_size = 0.2, # 20% used for testing
    seed = 42, # Seed for database split
    stratify_by_column = "label" # Keep label balance after split
)

# Remake DatasetDict with new split, since original had no test data labels
sst2_split = DatasetDict({
    "train": train_test_split["train"],
    "validation": sst2["validation"],
    "test": train_test_split["test"]
})

# Tokenisation of data

In [7]:
# Need model type for tokenisation.
model_name = "distilbert-base-uncased"

In [15]:
# Create the tokenizer for DistilBERT
tokenizer = AutoTokenizer.from_pretrained(model_name)

`tokenize` function:

- Takes:
  - `batch`: dictionary batch of rows from dataset

- Returns:
  - `tokenizer`: BertTokenizer containing numerical token IDs


Takes a batch of dataset rows and converts the sentences into token IDs. Also removed sentences longer than 128 tokens.

In [9]:
def tokenize(batch):
    return tokenizer(
        batch["sentence"], # select sentence text from dataset
        truncation=True, # cuts off text if it is too long
        max_length=128 # limits each sentence to 128 tokens. enough for SST2 due to short sentences
    )

In [16]:
# Applies tokenize function to each sentence
tokenized_sst2 = sst2_split.map(tokenize, batched=True)

# Remove columns no longer needed for training such as sentences, index.
tokenized_sst2 = tokenized_sst2.remove_columns(["sentence", "idx"])
# Rename column label to labels for compatibility
tokenized_sst2 = tokenized_sst2.rename_column("label", "labels")
# Return Pytorch tensor for tokenized_sst2
tokenized_sst2.set_format("torch")

# Setup Evaluation Metrics

`evaluation_metrics` function:

- Takes:
  - `evaluation_prediction`: 

- Returns:
  - `something`: 




In [ ]:
def evaluation_metrics(evaluation_prediciton):
    